# Train YOLO-pose — YOLO26 Large Pose

**Dataset:** `KLTN_POSE_V6_CLEAN` (4 keypoint TL→TR→BR→BL, 4 lớp, `kpt_shape [4,3]`, `flip_idx [1,0,3,2]`)

**Cách dùng:** Add Input file `.zip` dataset → Run All. Kết quả tải về ở `training_results_yolo26l.zip`.

| | |
|---|---|
| model | `yolo26l-pose.pt` |
| epochs / patience | 200 / 50 |
| batch / imgsz | 16 / 640 |
| dropout / weight_decay | 0.25 / 0.0005 |

> `imgsz=640` phải giữ nguyên khi suy diễn — chạy lệch scale làm tụt độ chính xác.


In [1]:
# 1. CÀI ĐẶT MÔI TRƯỜNG VÀ TẮT LOGGING CHỐNG KẸT
!pip install -q ultralytics
import os, gc, glob, shutil, re, collections
from ultralytics import YOLO

os.environ['WANDB_DISABLED'] = 'true'
os.environ['WANDB_MODE'] = 'dryrun'
os.environ['CLEARML_WEB_HOST'] = ''
print('[OK] Moi truong san sang')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 22.7 MB/s eta 0:00:00a 0:00:01
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
[OK] Moi truong san sang


In [2]:
# 2. TÌM DATASET + FIX ĐƯỜNG DẪN KAGGLE
yaml_paths = glob.glob('/kaggle/input/**/dataset.yaml', recursive=True)

if not yaml_paths:
    raise FileNotFoundError(
        'Khong tim thay dataset.yaml. Hay Add Input file .zip dataset vao notebook truoc!')

dataset_yaml = yaml_paths[0]
dataset_dir  = os.path.dirname(dataset_yaml)
print(f'[OK] Tim thay dataset tai: {dataset_yaml}')

with open(dataset_yaml, 'r', encoding='utf-8') as f:
    yaml_content = f.read()

# Ep path: ve dung thu muc thuc te tren Kaggle (path trong zip la duong dan may local)
new_yaml = re.sub(r'path:.*', f'path: {dataset_dir}', yaml_content)
working_yaml = '/kaggle/working/dataset.yaml'
with open(working_yaml, 'w', encoding='utf-8') as f:
    f.write(new_yaml)

print(f'[OK] Da fix path -> {working_yaml}\n')
print(new_yaml)

[OK] Tim thay dataset tai: /kaggle/input/datasets/nonlam123/dataposse-finnal/dataset.yaml
[OK] Da fix path -> /kaggle/working/dataset.yaml

# KLTN Pose V6 - CLEAN (split TRUOC, augment CHI train -> khong data leak)
path: /kaggle/input/datasets/nonlam123/dataposse-finnal
train: train/images
val: val/images
test: test/images

kpt_shape: [4, 3]
flip_idx: [1, 0, 3, 2]

names:
  0: anten-4G
  1: anten-5G
  2: rrh
  3: rru
nc: 4



In [3]:
# 3. KIỂM TRA DATASET TRƯỚC KHI TRAIN (chan loi "0 nhan" / sai format)
# Pose 4 diem: moi dong label phai co 1(class) + 4(box) + 4*3(kp) = 17 truong.
NUM_KP, EXP = 4, 1 + 4 + 4 * 3

assert 'kpt_shape' in new_yaml, 'THIEU kpt_shape trong dataset.yaml -> khong train pose duoc'
assert 'flip_idx'  in new_yaml, 'THIEU flip_idx -> fliplr se lam LOAN thu tu keypoint'

tong_obj = 0
for sp in ('train', 'val', 'test'):
    img_dir = os.path.join(dataset_dir, sp, 'images')
    lbl_dir = os.path.join(dataset_dir, sp, 'labels')
    if not os.path.isdir(img_dir):
        print(f'[--] khong co split "{sp}"'); continue
    imgs = [f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    lbls = glob.glob(os.path.join(lbl_dir, '*.txt'))
    cd, bad, obj, suybien = collections.Counter(), 0, 0, 0
    for lf in lbls:
        for ln in open(lf):
            p = ln.split()
            if not p:
                continue
            if len(p) != EXP:
                bad += 1; continue
            cd[int(p[0])] += 1; obj += 1
            # 4 keypoint trung nhau -> tu giac suy bien, solvePnP vo nghia
            pts = {(round(float(p[5 + k*3]), 5), round(float(p[6 + k*3]), 5)) for k in range(NUM_KP)}
            if len(pts) < NUM_KP:
                suybien += 1
    tong_obj += obj
    print(f'[{sp:5}] {len(imgs):4} anh | {len(lbls):4} label | {obj:5} object | '
          f'sai format: {bad} | keypoint trung nhau: {suybien} | theo lop: {dict(sorted(cd.items()))}')

assert tong_obj > 0, 'DATASET RONG -> kiem tra lai buoc split/convert'
print('\n[OK] Dataset hop le, bat dau train duoc.')

[train]  676 anh |  676 label |  1768 object | sai format: 0 | keypoint trung nhau: 0 | theo lop: {0: 1204, 1: 168, 2: 132, 3: 264}
[val  ]   21 anh |   21 label |    69 object | sai format: 0 | keypoint trung nhau: 0 | theo lop: {0: 48, 1: 7, 2: 2, 3: 12}
[test ]   21 anh |   21 label |    56 object | sai format: 0 | keypoint trung nhau: 0 | theo lop: {0: 43, 1: 6, 2: 1, 3: 6}

[OK] Dataset hop le, bat dau train duoc.


In [4]:
# 4. HUẤN LUYỆN YOLO26 Large Pose
model_variant = 'yolo26l-pose.pt'

epochs       = 200
patience     = 50          # dung som neu 50 epoch khong cai thien
batch_size   = 16
img_size     = 640         # PHAI trung voi imgsz luc suy dien
dropout      = 0.25
weight_decay = 0.0005

# Bat Dual-GPU neu Kaggle dang cap 2x T4
device = '0,1' if os.path.exists('/dev/nvidia1') else '0'
print(f'model={model_variant} | device={device} | batch={batch_size} | imgsz={img_size}')

model = YOLO(model_variant)
results = model.train(
    data=working_yaml,
    epochs=epochs,
    patience=patience,
    batch=batch_size,
    imgsz=img_size,
    device=device,
    dropout=dropout,
    weight_decay=weight_decay,
    project='/kaggle/working/runs',
    name='pose_model',      # giu ten nay de khop script chay local
    exist_ok=True,
    plots=True,
)
print('[OK] Train xong.')

model=yolo26l-pose.pt | device=0,1 | batch=16 | imgsz=640
Ultralytics 8.4.107 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/dataset.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.25, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_

In [6]:
# 5. ĐÁNH GIÁ TRÊN TEST SPLIT + NÉN KẾT QUẢ
best = '/kaggle/working/runs/pose_model/weights/best.pt'
m = YOLO(best)

try:
    r = m.val(data=working_yaml, split='test', imgsz=img_size, device=device.split(',')[0], plots=False)
    print(f'\n=== TEST ({model_variant}) ===')
    print(f'  Box  mAP50={r.box.map50:.4f}  mAP50-95={r.box.map:.4f}')
    print(f'  Pose mAP50={r.pose.map50:.4f}  mAP50-95={r.pose.map:.4f}')
except Exception as e:
    print('Bo qua val test:', e)

zip_path = shutil.make_archive('/kaggle/working/training_results_yolo26l', 'zip', '/kaggle/working/runs')
print(f'\n[OK] Da nen: {zip_path}  ({os.path.getsize(zip_path)/1024/1024:.1f} MB)')
print('=> Tai file training_results_yolo26l.zip ve may.')

for _v in ('model', 'm'):          # don RAM; dung globals() de khong loi neu cell 4 chua chay
    globals().pop(_v, None)
gc.collect()

Ultralytics 8.4.107 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO26l-pose summary (fused): 200 layers, 25,599,420 parameters, 0 gradients, 89.8 GFLOPs
val: Fast image access ✅ (ping: 2.0±0.5 ms, read: 132.6±89.9 MB/s, size: 167.9 KB)
val: Scanning /kaggle/input/datasets/nonlam123/dataposse-finnal/test/labels... 21 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 21/21 524.3it/s 0.0s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/nonlam123/dataposse-finnal/test is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Pose(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.5it/s 1.3s3.8s
                   all         21         56      0.563      0.816       0.83      0.642      0.563      0.816      0.804      0.792
              anten-4G         21         43      0.737       0.93      0.875      0.695      0.737       0.93      0.875      0.847
              anten-5G      

9914